# 从 Curve 到 Protein：RFdiffusion 魔改 ProtPainter 完整演示

这是一份面向初学者的完整 notebook，目标是把下面几个问题一次讲清楚：

1. 扩散模型是什么。
2. 蛋白质在代码里是怎么表示的。
3. 这个项目如何把一条 3D 曲线变成一个蛋白质结构。
4. `lamda_0`、`lamda_1`、`lamda_2` 的作用是什么。
5. 无条件生成和有条件生成的结果有什么区别。

## 1. 扩散模型的直观理解

一个非常朴素的理解方式是：

- 前向过程：不断给一个结构加噪声，直到它变得很乱。
- 反向过程：训练一个模型，学习如何一步一步把噪声去掉。
- 生成时：从随机噪声出发，沿着反向过程逐步得到一个合理结构。

在图像任务里，扩散模型生成图片；在这个项目里，扩散模型生成蛋白质骨架。

## 2. 这个项目在做什么

这个项目不是“凭空生成一个蛋白”，而是“给定一个几何条件，再生成一个蛋白”。

在你的使用场景中，几何条件是一条 3D curve。整个流程可以简化成：

1. 输入一条 3D 曲线。
2. 用它生成像蛋白主链的 sketch。
3. 用 RFdiffusion 在 sketch 条件下生成蛋白骨架。

所以它更像“按草图生成蛋白”，而不是“自由发挥生成蛋白”。

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlopen

DEFAULT_DEMO_REPO_URL = os.environ.get(
    "RFDIFFUSION_DEMO_REPO_URL",
    "https://github.com/wewewexiao2008/rfdiffusion_drawprotein_demo.git",
)
DEFAULT_DEMO_REPO_DIR = os.environ.get("RFDIFFUSION_DEMO_REPO_DIR", "rfdiffusion_drawprotein_demo")
OFFICIAL_RFDIFFUSION_REPO_URL = os.environ.get(
    "OFFICIAL_RFDIFFUSION_REPO_URL",
    "https://github.com/RosettaCommons/RFdiffusion.git",
)
OFFICIAL_RFDIFFUSION_DIR = os.environ.get("OFFICIAL_RFDIFFUSION_DIR", "RFdiffusion")
REQUIRED_MODEL_URLS = {
    "Base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt",
    "Complex_base_ckpt.pt": "http://files.ipd.uw.edu/pub/RFdiffusion/e29311f6f1bf1af907f9ef9f44b8328b/Complex_base_ckpt.pt",
}

# Keep the setup logic local to this cell so the notebook stays easy to rerun.
def find_existing_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    candidates.append(Path.cwd() / DEFAULT_DEMO_REPO_DIR)
    for path in candidates:
        if (path / "scripts" / "draw_protein.py").exists() and (path / "requirements.txt").exists():
            return path.resolve()
    return None


def ensure_demo_repo() -> Path:
    existing = find_existing_project_root()
    if existing is not None:
        return existing
    clone_target = (Path.cwd() / DEFAULT_DEMO_REPO_DIR).resolve()
    if clone_target.exists() and any(clone_target.iterdir()):
        raise FileExistsError(
            f"目标目录 {clone_target} 已存在但不是项目根目录，请清理后重试，或设置 RFDIFFUSION_DEMO_REPO_DIR。"
        )
    print(f"未检测到 demo 仓库，开始 clone: {DEFAULT_DEMO_REPO_URL}")
    run(["git", "clone", "--depth", "1", DEFAULT_DEMO_REPO_URL, str(clone_target)])
    return clone_target


def run(cmd, env=None, cwd=None):
    print("[执行]", " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, env=env, cwd=cwd)


def ensure_current_kernel_packages():
    missing = []
    for package, module_name in [("numpy", "numpy"), ("matplotlib", "matplotlib"), ("biopython", "Bio")]:
        try:
            importlib.import_module(module_name)
        except ImportError:
            missing.append(package)
    if missing:
        run([sys.executable, "-m", "pip", "install", *missing])


def find_official_rfdiffusion_root(project_root: Path):
    candidates = []
    env_override = os.environ.get("OFFICIAL_RFDIFFUSION_ROOT")
    if env_override:
        candidates.append(Path(env_override).expanduser())
    candidates.append(project_root.parent / OFFICIAL_RFDIFFUSION_DIR)
    for path in candidates:
        if (path / "env" / "SE3nv.yml").exists():
            return path.resolve()
    return None


def ensure_official_rfdiffusion_root(project_root: Path) -> Path:
    existing = find_official_rfdiffusion_root(project_root)
    if existing is not None:
        return existing
    clone_target = (project_root.parent / OFFICIAL_RFDIFFUSION_DIR).resolve()
    if clone_target.exists() and any(clone_target.iterdir()):
        raise FileExistsError(
            f"目标目录 {clone_target} 已存在但不是官方 RFdiffusion 根目录，请清理后重试，或设置 OFFICIAL_RFDIFFUSION_ROOT。"
        )
    print(f"未检测到官方 RFdiffusion 仓库，开始 clone: {OFFICIAL_RFDIFFUSION_REPO_URL}")
    run(["git", "clone", "--depth", "1", OFFICIAL_RFDIFFUSION_REPO_URL, str(clone_target)])
    return clone_target


def ensure_micromamba() -> Path:
    existing = shutil.which("micromamba")
    if existing:
        return Path(existing)
    local_bin = Path.home() / ".local" / "bin"
    local_bin.mkdir(parents=True, exist_ok=True)
    micromamba_bin = local_bin / "micromamba"
    if not micromamba_bin.exists():
        print("未检测到 micromamba，开始安装到 ~/.local/bin ...")
        archive_cmd = (
            f'curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest '
            f'| tar -xj -C "{local_bin}" --strip-components=1 bin/micromamba'
        )
        run(["bash", "-lc", archive_cmd])
    if not micromamba_bin.exists():
        raise FileNotFoundError("micromamba 安装失败，请检查网络或 PATH。")
    return micromamba_bin


def download_file(url: str, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    with urlopen(url) as response, open(partial, "wb") as handle:
        shutil.copyfileobj(response, handle)
    partial.replace(destination)


def ensure_required_weights(project_root: Path):
    models_dir = project_root / "models"
    models_dir.mkdir(parents=True, exist_ok=True)
    for filename, url in REQUIRED_MODEL_URLS.items():
        target = models_dir / filename
        if target.exists() and target.stat().st_size > 0:
            print(f"检测到模型权重: {target}")
            continue
        print(f"下载模型权重: {filename}")
        download_file(url, target)


project_root = ensure_demo_repo()
official_root = ensure_official_rfdiffusion_root(project_root)
micromamba_bin = ensure_micromamba()

env = os.environ.copy()
env["PATH"] = f"{micromamba_bin.parent}:{env.get('PATH', '')}"
env["MAMBA_ROOT_PREFIX"] = str(project_root / ".mamba")

env_dir = project_root / ".notebook_env"
env_python = env_dir / "bin" / "python"
env_stamp = env_dir / ".bootstrap_complete"
requirements_file = project_root / "requirements.txt"

print(f"项目根目录: {project_root}")
print(f"Demo 仓库来源: {DEFAULT_DEMO_REPO_URL}")
print(f"官方 RFdiffusion 仓库来源: {OFFICIAL_RFDIFFUSION_REPO_URL}")
print(f"官方 RFdiffusion 根目录: {official_root}")
print(f"Notebook 托管环境目录: {env_dir}")

ensure_current_kernel_packages()
ensure_required_weights(project_root)

if not env_python.exists():
    run([
        str(micromamba_bin),
        "create",
        "-y",
        "-p",
        str(env_dir),
        "-f",
        str(official_root / "env" / "SE3nv.yml"),
    ], env=env)

if not env_stamp.exists():
    se3_root = official_root / "env" / "SE3Transformer"
    run([
        str(env_python),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-r",
        str(se3_root / "requirements.txt"),
    ], env=env)
    run([str(env_python), "setup.py", "install"], cwd=str(se3_root), env=env)

    run([
        str(env_python),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-r",
        str(requirements_file),
    ], env=env)

    site_packages = subprocess.check_output(
        [str(env_python), "-c", "import site; print(site.getsitepackages()[0])"],
        text=True,
        env=env,
    ).strip()
    pth_file = Path(site_packages) / "rfdiffusion_draw_protein_minimal.pth"
    pth_file.write_text(f"{project_root}\n")
    env_stamp.write_text("ok\n")
else:
    print("检测到已完成的 notebook 环境，跳过依赖安装。")

ENV_PYTHON = env_python
os.environ["RFDIFFUSION_ENV_PYTHON"] = str(ENV_PYTHON)

def run_in_env(args, cwd=None, extra_env=None):
    call_env = env.copy()
    if extra_env:
        call_env.update(extra_env)
    cmd = [str(ENV_PYTHON), *map(str, args)]
    print("[环境执行]", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=str(cwd or project_root), env=call_env)


print("\n环境配置完成。")
print("1. notebook 会继续使用当前 kernel 做展示和可视化。")
print(f"2. 所有推理命令都会通过 {ENV_PYTHON} 执行。")
print("3. setup 会自动确保 Base_ckpt.pt 和 Complex_base_ckpt.pt 已下载到 models/。")

In [ ]:
from pathlib import Path
import pickle
import subprocess

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from Bio.PDB import PDBParser

plt.rcParams["figure.figsize"] = (6, 5)

def locate_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for path in candidates:
        if (path / "scripts" / "draw_protein.py").exists():
            return path
    raise FileNotFoundError("Could not locate the project root.")

PROJECT_ROOT = locate_project_root()
RUN_DIR = PROJECT_ROOT / "runs" / "modatest"
UNCOND_RUN = PROJECT_ROOT / "runs" / "unconditional_demo"

curve_path = PROJECT_ROOT / "HHH_ems_50_npy" / "HHH_ems_00001.npy"
curve_copy_path = RUN_DIR / "HHH_ems_00001.npy"
sketch_path = RUN_DIR / "sketchs" / "HHH_ems_00001-51.npy"
design_pdb_path = RUN_DIR / "designs" / "HHH_ems_00001-51_0.pdb"
traj_path = RUN_DIR / "rfdiffusion" / "traj" / "HHH_ems_00001-51_0_Xt-1_traj.pdb"
trb_path = RUN_DIR / "rfdiffusion" / "HHH_ems_00001-51_0.trb"
uncond_design_path = UNCOND_RUN / "designs" / "51_0.pdb"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("curve exists:", curve_path.exists())
print("copied curve exists:", curve_copy_path.exists())
print("sketch exists:", sketch_path.exists())
print("design exists:", design_pdb_path.exists())
print("trajectory exists:", traj_path.exists())
print("trb exists:", trb_path.exists())
print("unconditional design exists:", uncond_design_path.exists())

## 3. 蛋白质的数据结构

PDB 文件通常按照下面这层结构组织：

`Structure -> Model -> Chain -> Residue -> Atom`

对初学者来说，这一层最重要：

- 一个蛋白由很多残基（residue）组成。
- 每个残基由很多原子（atom）组成。
- 在结构设计里，C-alpha（CA）原子很重要，因为它决定了主链的大致轨迹。

而在这个项目里，最常见的数值表示是：

- `curve/sketch`: `L x 3`
- `CA coordinates`: `L x 3`
- `all-atom representation`（模型内部）: 类似 `L x 14 x 3`

所以你可以把它理解成：外部输入是点云轨迹，内部模型再把它扩展成更完整的蛋白骨架表示。

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("design", str(design_pdb_path))

models = list(structure)
chains = list(models[0])
residues = list(chains[0].get_residues())
atoms = list(structure.get_atoms())

print("Number of models:", len(models))
print("Number of chains in model 0:", len(chains))
print("Number of residues in chain A:", len(residues))
print("Number of atoms in the whole structure:", len(atoms))
print("First residue name:", residues[0].get_resname())
print("First residue atom names:", [atom.get_name() for atom in list(residues[0])])

In [ ]:
def load_ca_from_pdb(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", str(pdb_path))
    coords = []
    for atom in structure.get_atoms():
        if atom.get_name() == "CA":
            coords.append(atom.get_coord())
    return np.array(coords)

def plot_trace(ax, coords, title, color):
    ax.plot(coords[:, 0], coords[:, 1], coords[:, 2], color=color, linewidth=2)
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], color=color, s=10)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")

curve = np.loadtxt(curve_path, delimiter=",")[:, :3]
curve_copy = np.loadtxt(curve_copy_path, delimiter=",")[:, :3]
sketch = np.loadtxt(sketch_path, delimiter=",")[:, :3]
design_ca = load_ca_from_pdb(design_pdb_path)
uncond_design = load_ca_from_pdb(uncond_design_path) if uncond_design_path.exists() else None

print("input curve shape:", curve.shape)
print("copied curve shape:", curve_copy.shape)
print("sketch shape:", sketch.shape)
print("design CA shape:", design_ca.shape)
print("unconditional CA shape:", None if uncond_design is None else uncond_design.shape)

## 4. 扩散模型的基础公式

前向扩散可以写成：

$$
q(x_t \mid x_{t-1}) = \mathcal{N}(\sqrt{1-\beta_t}x_{t-1}, \beta_t I)
$$

意思是：每一步都往结构里加入一点噪声。

反向过程由模型学习：

$$
p_\theta(x_{t-1} \mid x_t, c) = \mathcal{N}(\mu_\theta(x_t, c, t), \sigma_t I)
$$

其中：

- $x_t$ 是当前带噪结构
- $c$ 是条件信息
- $\mu_\theta$ 是模型预测的下一步去噪方向

在这个项目的 curve-conditioned 版本里，可以粗略把更新理解成：

$$
\mu_t = a_t\,p(x_0) + b_t\,x_t + c_t\,cX
$$

这里：

- $p(x_0)$ 是模型预测的干净结构
- $x_t$ 是当前带噪结构
- $cX$ 是外部条件，也就是 curve/sketch

这也是这个项目为什么既能“生成”，又能“被条件引导”的根本原因。

## 5. 这个项目中的真实流程

这个 notebook 实际通过 setup 单元里定义的 `ENV_PYTHON` 去运行：

```bash
$ENV_PYTHON scripts/draw_protein.py -c ./HHH_ems_50_npy/HHH_ems_00001.npy -n modatest -l 50
```

这条命令做了两件事：

1. `curve -> sketch`
2. `sketch -> RFdiffusion protein`

先看输入、中间结果和最终输出。

In [ ]:
fig = plt.figure(figsize=(15, 4.5))

ax1 = fig.add_subplot(131, projection="3d")
plot_trace(ax1, curve, "Step 1: Input Curve", "tab:blue")

ax2 = fig.add_subplot(132, projection="3d")
plot_trace(ax2, sketch, "Step 2: Sketch", "tab:orange")

ax3 = fig.add_subplot(133, projection="3d")
plot_trace(ax3, design_ca, "Step 3: Designed Protein", "tab:green")

plt.tight_layout()
plt.show()

## 6. `.trb` 文件里有什么

RFdiffusion 会把每次设计的元信息写进 `.trb`。它很适合用来回答“这次到底用了什么参数”和“模型在过程中的置信度如何变化”。

In [ ]:
with open(trb_path, "rb") as f:
    trb = pickle.load(f)

print("Keys in trb:")
for key in sorted(trb.keys()):
    print("-", key)

print("\nSelected inference config:")
print("method:", trb["config"]["inference"].get("method"))
print("lamda_0:", trb["config"]["inference"].get("lamda_0"))
print("lamda_1:", trb["config"]["inference"].get("lamda_1"))
print("lamda_2:", trb["config"]["inference"].get("lamda_2"))
print("contigs:", trb["config"]["contigmap"].get("contigs"))
print("device:", trb.get("device"))
print("plddt shape:", trb["plddt"].shape)

In [ ]:
mean_plddt_by_step = trb["plddt"].mean(axis=1)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(len(mean_plddt_by_step), 0, -1), mean_plddt_by_step, marker="o")
plt.xlabel("Reverse diffusion step")
plt.ylabel("Mean pLDDT")
plt.title("Confidence over the reverse trajectory")
plt.grid(alpha=0.3)
plt.show()

## 7. 动画：扩散去噪轨迹

下面的动画直接来自你这次运行保存的 trajectory 文件。

这部分最适合给第一次接触扩散模型的人看，因为它把“逐步去噪”变成了肉眼可见的过程。

In [ ]:
def load_ca_trajectory(pdb_path):
    text = Path(pdb_path).read_text()
    blocks = text.split("ENDMDL")
    frames = []
    for block in blocks:
        coords = []
        for line in block.splitlines():
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                x = float(line[30:38])
                y = float(line[38:46])
                z = float(line[46:54])
                coords.append([x, y, z])
        if coords:
            frames.append(np.array(coords))
    return frames

traj_frames = load_ca_trajectory(traj_path)
print("number of trajectory frames:", len(traj_frames))
print("frame shape:", traj_frames[0].shape)

In [ ]:
frames_to_show = traj_frames[::2] if len(traj_frames) > 25 else traj_frames
all_points = np.concatenate(frames_to_show, axis=0)
mins = all_points.min(axis=0)
maxs = all_points.max(axis=0)

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection="3d")
line, = ax.plot([], [], [], color="tab:red", linewidth=2)

ax.set_xlim(mins[0], maxs[0])
ax.set_ylim(mins[1], maxs[1])
ax.set_zlim(mins[2], maxs[2])
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

def update(frame_idx):
    coords = frames_to_show[frame_idx]
    line.set_data(coords[:, 0], coords[:, 1])
    line.set_3d_properties(coords[:, 2])
    ax.collections.clear()
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], color="tab:red", s=10)
    ax.set_title(f"Reverse diffusion frame {frame_idx + 1}/{len(frames_to_show)}")
    return line,

anim = FuncAnimation(fig, update, frames=len(frames_to_show), interval=250, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

## 8. `lamda_0`, `lamda_1`, `lamda_2` 的作用

在这个项目的 curve-conditioned 版本里，下一步更新均值可以粗略理解成：

$$
\mu_t = a_t\,p(x_0) + b_t\,x_t + c_t\,cX
$$

三个 `lamda` 参数主要影响 `b_t` 和 `c_t`，也就是：

- 当前噪声结构 `x_t` 要占多大权重
- 条件 sketch `cX` 要占多大权重

非常适合初学者的直观总结是：

- `lamda_0`：总的条件强度。越大，结果通常越愿意贴近 sketch。
- `lamda_1`：对二级结构差异的敏感度。这里主要和 helix 比例差相关。
- `lamda_2`：阻尼项。它会抑制条件项过强，避免更新过猛。

换句话说，它们都在调节一个平衡：

**是更忠实地跟随条件，还是给模型更多自由去生成自然结构。**

## 9. 无条件生成 vs 有条件生成

现在看另一个非常适合展示的问题：

- **unconditional generation**：不给几何条件，只给长度。
- **conditioned generation**：给几何条件，让模型沿着这个条件生成。

它们最直观的区别是：

- unconditional 更自由，但可控性更弱。
- conditioned 更像“按草图生成”。

In [ ]:
RUN_UNCONDITIONAL = False

if RUN_UNCONDITIONAL:
    args = [
        "scripts/draw_protein.py",
        "-n", "unconditional_demo",
        "-t", "unconditional",
        "-l", "51",
        "-num", "1",
    ]
    print("Running in managed env:", " ".join([str(ENV_PYTHON), *args]))
    run_in_env(args, cwd=PROJECT_ROOT)
else:
    print("Set RUN_UNCONDITIONAL = True if you want to generate the unconditional example.")

In [ ]:
def radius_of_gyration(coords):
    center = coords.mean(axis=0)
    return np.sqrt(((coords - center) ** 2).sum(axis=1).mean())

print("Conditioned design length:", len(design_ca))
print("Conditioned design Rg:", round(radius_of_gyration(design_ca), 3))

if uncond_design is not None:
    print("Unconditional design length:", len(uncond_design))
    print("Unconditional design Rg:", round(radius_of_gyration(uncond_design), 3))
else:
    print("Unconditional result not available yet.")

In [ ]:
fig = plt.figure(figsize=(15, 4.5))

ax1 = fig.add_subplot(131, projection="3d")
plot_trace(ax1, sketch, "Conditioning Sketch", "tab:orange")

ax2 = fig.add_subplot(132, projection="3d")
plot_trace(ax2, design_ca, "Conditioned Design", "tab:green")

ax3 = fig.add_subplot(133, projection="3d")
if uncond_design is not None:
        plot_trace(ax3, uncond_design, "Unconditional Design", "tab:purple")
else:
    ax3.text(0.5, 0.5, 0.5, "Run the unconditional cell first", ha="center")
    ax3.set_title("Unconditional Design")

plt.tight_layout()
plt.show()

## 10. 如何向初学者解释两者差别

一个非常适合展示时说的话术是：

- 无条件生成：我只告诉模型“做一个长度差不多的蛋白”，其他让它自己决定。
- 有条件生成：我告诉模型“沿着这条几何路径做一个蛋白”。

所以：

- unconditional 更像自由采样
- conditioned 更像几何约束设计

## 11. 总结

如果只用一句话总结这个项目，可以这样说：

> 给定一条空间曲线，先把它变成蛋白 sketch，再用条件扩散模型逐步去噪，得到一个符合这条几何轨迹的蛋白质骨架。

这份 notebook 已经覆盖了：

- 扩散模型的基本直觉
- 蛋白质的数据结构
- curve -> sketch -> protein 的真实流程
- `lamda_0/1/2` 的直观含义
- conditioned 与 unconditional 的对比

如果后面要继续升级展示材料，最自然的下一步是把这里的图和动画整理成正式 slides。